# Homework Starter — Stage 04: Data Acquisition and Ingestion
Name: Po-Wei Su
Date: 2026-08-18

## Objectives
- API ingestion with secrets in `.env`
- Scrape a permitted public table
- Validate and save raw data to `data/raw/`

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install pandas
# !pip install requests
# !pip install yfinance
# !pip install python-dotenv
# !pip install beautifulsoup4

In [2]:
# --- run me first: makes this notebook work wherever it lives ---
from pathlib import Path
import os, sys

if Path.cwd().name == "notebooks":
    os.chdir("..")  # project/notebooks -> project

ROOT = Path.cwd()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("working from:", ROOT.name)

# --- files this notebook needs ---
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

working from: project
Looking in: /Users/albert/bootcamp_Albert_Su/project

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [3]:
import os, pathlib, datetime as dt
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

RAW = pathlib.Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
load_dotenv(); print('ALPHAVANTAGE_API_KEY loaded?', bool(os.getenv('ALPHAVANTAGE_API_KEY')))

ALPHAVANTAGE_API_KEY loaded? False


## Helpers (use or modify)

In [4]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

def save_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join([f"{k}-{v}" for k,v in meta.items()])
    path = RAW / f"{prefix}_{mid}_{ts()}.csv"
    df.to_csv(path, index=False)
    print('Saved', path)
    return path

def validate(df: pd.DataFrame, required):
    missing = [c for c in required if c not in df.columns]
    return {'missing': missing, 'shape': df.shape, 'na_total': int(df.isna().sum().sum())}

## Part 1 — API Pull (Required)
Choose an endpoint (e.g., Alpha Vantage or use `yfinance` fallback).

In [5]:
SYMBOL = 'SPY'
USE_ALPHA = bool(os.getenv('ALPHAVANTAGE_API_KEY'))
if USE_ALPHA:
    url = 'https://www.alphavantage.co/query'
    params = {'function':'TIME_SERIES_DAILY_ADJUSTED','symbol':SYMBOL,'outputsize':'compact','apikey':os.getenv('ALPHAVANTAGE_API_KEY')}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    key = [k for k in js if 'Time Series' in k][0]
    df_api = pd.DataFrame(js[key]).T.reset_index().rename(columns={'index':'date','5. adjusted close':'adj_close'})[['date','adj_close']]
    df_api['date'] = pd.to_datetime(df_api['date']); df_api['adj_close'] = pd.to_numeric(df_api['adj_close'])
else:
    import yfinance as yf
    # df_api = yf.download(SYMBOL, period='3mo', interval='1d').reset_index()[['Date','Adj Close']]
    df_api = yf.download(
    SYMBOL,
    period='3mo',
    interval='1d',
    auto_adjust=False,
    multi_level_index=False
        ).reset_index()[['Date', 'Adj Close']]
    
    df_api.columns = ['date','adj_close']

v_api = validate(df_api, ['date','adj_close']); v_api

[*********************100%***********************]  1 of 1 completed

{'missing': [], 'shape': (64, 2), 'na_total': 0}

In [6]:
SCRAPE_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent':'AFE-Homework/1.0'}

try:
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, 'html.parser')

    table = soup.find('table', id='constituents')

    rows = [
        [c.get_text(strip=True) for c in tr.find_all(['th','td'])]
        for tr in table.find_all('tr')
    ]

    header, *data = [r for r in rows if r]
    df_scrape = pd.DataFrame(data, columns=header)

except Exception as e:
    print('Scrape failed, using inline demo table:', e)
    html = '<table><tr><th>Ticker</th><th>Price</th></tr><tr><td>AAA</td><td>101.2</td></tr></table>'
    soup = BeautifulSoup(html, 'html.parser')
    rows = [[c.get_text(strip=True) for c in tr.find_all(['th','td'])] for tr in soup.find_all('tr')]
    header, *data = [r for r in rows if r]
    df_scrape = pd.DataFrame(data, columns=header)

if 'Date added' in df_scrape.columns:
    df_scrape['Date added'] = pd.to_datetime(
        df_scrape['Date added'],
        errors='coerce'
    )

v_scrape = validate(df_scrape, list(df_scrape.columns))
print(v_scrape)
df_scrape.dtypes

{'missing': [], 'shape': (503, 8), 'na_total': 0}


Symbol                           object
Security                         object
GICSSector                       object
GICS Sub-Industry                object
Headquarters Location            object
Date added               datetime64[ns]
CIK                              object
Founded                          object
dtype: object

In [7]:
_ = save_csv(df_api.sort_values('date'), prefix='api', source='alpha' if USE_ALPHA else 'yfinance', symbol=SYMBOL)

Saved data/raw/api_source-yfinance_symbol-SPY_20260819-115821.csv


## Part 2 — Scrape a Public Table (Required)
Replace `SCRAPE_URL` with a permitted page containing a simple table.

In [8]:
_ = save_csv(df_scrape, prefix='scrape', site='wikipedia', table='sp500')

Saved data/raw/scrape_site-wikipedia_table-sp500_20260819-115821.csv


## Documentation

- **API Source:** Yahoo Finance via `yfinance`
  - Symbol: SPY
  - Period: 3 months
  - Interval: 1 day
  - `auto_adjust=False`

- **Scrape Source:** Wikipedia — List of S&P 500 companies
  - URL: https://en.wikipedia.org/wiki/List_of_S%26P_500_companies
  - Table: S&P 500 constituents (`id="constituents"`)

- **Validation Logic:**
  - Checked required columns.
  - Checked DataFrame shape.
  - Checked total missing values.
  - Parsed API dates as datetime and adjusted close as numeric.
  - Parsed the scraped `Date added` column as datetime.

- **Assumptions & Risks:**
  - Yahoo Finance data availability and schema may change.
  - Wikipedia table structure or column names may change.
  - The HTML selector `id="constituents"` may stop working if the page structure changes.
  - Network failures may interrupt data acquisition.
  - Raw files are timestamped so each ingestion run can be reproduced and traced.

- **Secrets:** `.env` is stored locally and should not be committed to Git.

# Stage 05 — Data Storage

Use the SPY market data acquired in Stage 04 as the input for reproducible storage and validation.

In [9]:
import pandas as pd

spy_files = sorted((ROOT / "data" / "raw").glob(
    "api_source-yfinance_symbol-SPY_*.csv"
))

if not spy_files:
    raise FileNotFoundError("No Stage 04 SPY raw CSV found.")

spy_raw_path = spy_files[-1]
df = pd.read_csv(spy_raw_path, parse_dates=["date"])

print("Loaded:", spy_raw_path.name)
print("Shape:", df.shape)
print(df.dtypes)
df.head()


Loaded: api_source-yfinance_symbol-SPY_20260819-115821.csv
Shape: (64, 2)
date         datetime64[ns]
adj_close           float64
dtype: object


,date,adj_close
0,2026-05-19,731.844604
1,2026-05-20,739.345276
2,2026-05-21,740.811462
3,2026-05-22,743.723999
4,2026-05-26,748.661316


## Stage 05 — Store and Validate

Store the Stage 04 SPY raw data in Parquet format and verify that it reloads correctly.

In [10]:
import datetime as dt

processed_dir = ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

timestamp = dt.datetime.now().strftime("%Y%m%d-%H%M%S")
spy_parquet_path = processed_dir / f"spy_prices_{timestamp}.parquet"

df.to_parquet(spy_parquet_path, index=False)

df_parquet = pd.read_parquet(spy_parquet_path)

checks = {
    "shape_equal": df.shape == df_parquet.shape,
    "date_is_datetime": pd.api.types.is_datetime64_any_dtype(df_parquet["date"]),
    "adj_close_is_numeric": pd.api.types.is_numeric_dtype(df_parquet["adj_close"]),
}

print("Saved:", spy_parquet_path)
print("Validation:", checks)
df_parquet.head()


Saved: /Users/albert/bootcamp_Albert_Su/project/data/processed/spy_prices_20260819-115821.parquet
Validation: {'shape_equal': True, 'date_is_datetime': True, 'adj_close_is_numeric': True}


,date,adj_close
0,2026-05-19,731.844604
1,2026-05-20,739.345276
2,2026-05-21,740.811462
3,2026-05-22,743.723999
4,2026-05-26,748.661316
